# Experimentos 4 e 5 - Ajuste de learning rate e orçamento de épocas

Projeto: **Detecção de Faces para Anonimização de Imagens** (conformidade com a LGPD) - disciplina de Introdução à Ciência de Dados e Aprendizado de Máquina (IFSC).

Os experimentos 2 e 3 produziram uma anomalia: o fine-tuning completo (exp3) perdeu para o feature extraction (exp2), contra a expectativa de que mais parâmetros treináveis renderiam mais. As curvas de treinamento indicaram **subtreino**: com `lr0=1e-4`, a rede inteira se move devagar demais para 20 épocas (diagnóstico completo em `docs/experimentos.md`).

Este notebook testa as duas variáveis envolvidas, **uma de cada vez**:

- **Experimento 4** - configuração idêntica ao exp3, mudando apenas a learning rate: `lr0` de 1e-4 para **1e-3**. Se a hipótese de subtreino (H1) estiver certa, deve alcançar ou superar o exp2.
- **Experimento 5** - configuração idêntica ao exp4, mudando apenas o orçamento: **50 épocas** em vez de 20. As curvas anteriores não mostraram overfitting até a época 20; aqui verificamos se há ganho adicional e onde o desempenho satura ou começa a cair.

> **Colab:** selecione GPU (T4). Os dois treinos em sequência levam ~1h30–2h (em GPU local RTX 3060, ~1h).

## 1. Instalação das dependências

O Colab já traz OpenCV, Matplotlib e Pandas pré-instalados; instalamos apenas a biblioteca `ultralytics` (YOLO11) e confirmamos a versão e a disponibilidade de GPU. A checagem é feita diretamente com `torch.cuda.is_available()` - o utilitário `ultralytics.checks()` dispara subprocessos de inspeção do ambiente que podem travar em algumas máquinas.

In [ ]:
%pip install -q "ultralytics>=8.3.0"

import torch
import ultralytics

print('ultralytics', ultralytics.__version__)
print('GPU disponivel:', torch.cuda.is_available())

## 2. Parâmetros

Os parâmetros do dataset devem ser **idênticos aos dos notebooks anteriores**. As duas configurações de treino ficam na lista `CONFIGURACOES` - a única diferença entre elas é o número de épocas, e a única diferença de ambas para o exp3 é a learning rate.

Como nos demais experimentos com `lr0` controlada, fixamos `optimizer='AdamW'` (com `optimizer='auto'`, a Ultralytics ignora o `lr0` informado).

In [ ]:
import sys
from pathlib import Path

EM_COLAB = 'google.colab' in sys.modules

REPO_URL = 'https://github.com/ifsc-sj-projetos-ia/corrige-aqui'

if EM_COLAB:
    BASE_DIR = Path('/content/projeto-faces')
elif Path.cwd().name == 'notebooks':
    BASE_DIR = Path.cwd().parent
else:
    BASE_DIR = Path.cwd()

TRAIN_SIZE = 1500
VAL_FRACTION = 0.2
TEST_SIZE = 300
SEED = 42
IMGSZ = 640
LR0 = 0.001
OTIMIZADOR = 'AdamW'
BATCH = 8
WORKERS = 2

CONFIGURACOES = [
    {'nome': 'exp4_fine_tuning_lr1e3', 'epochs': 20},
    {'nome': 'exp5_fine_tuning_lr1e3_50ep', 'epochs': 50},
]

## 3. Obtenção e preparação do dataset

Mesmo procedimento dos notebooks anteriores. Com a mesma `SEED`, o subset e os splits são **exatamente os mesmos** dos outros experimentos.

In [ ]:
if EM_COLAB and not BASE_DIR.exists():
    !git clone "{REPO_URL}" "{BASE_DIR}"

DATA_DIR = BASE_DIR / 'data'
SUBSET_DIR = DATA_DIR / 'wider_subset'
DATASET_YAML = SUBSET_DIR / 'dataset.yaml'
SCRIPT_PREPARO = BASE_DIR / 'src' / 'prepare_dataset.py'

if not DATASET_YAML.exists():
    !"{sys.executable}" "{SCRIPT_PREPARO}" --data-dir "{DATA_DIR}" --train-size {TRAIN_SIZE} --val-fraction {VAL_FRACTION} --test-size {TEST_SIZE} --seed {SEED}

## 4. Treinamento das duas configurações

Pontos importantes:

- O modelo é **recriado do zero a cada configuração** (`YOLO('yolo11n.pt')` dentro do laço): cada experimento parte dos mesmos pesos pré-treinados do COCO.
- O exp5 **não é o exp4 continuado**: a learning rate decai linearmente até o fim do treino **em função do total de épocas**, então um run de 50 épocas tem trajetória de lr diferente de um run de 20 - são experimentos independentes.
- Após cada treino, o melhor checkpoint (`best.pt`, selecionado pela validação) é avaliado no conjunto de **teste**.
- Re-execuções são reprodutíveis: mesma seed e substituição da linha correspondente no CSV de métricas.

In [ ]:
from ultralytics import YOLO

resultados_teste_por_experimento = {}
for config in CONFIGURACOES:
    modelo = YOLO('yolo11n.pt')
    modelo.train(
        data=str(DATASET_YAML),
        epochs=config['epochs'],
        lr0=LR0,
        optimizer=OTIMIZADOR,
        imgsz=IMGSZ,
        batch=BATCH,
        workers=WORKERS,
        seed=SEED,
        project=str(BASE_DIR / 'models'),
        name=config['nome'],
        exist_ok=True,
    )
    melhor_modelo = YOLO(str(BASE_DIR / 'models' / config['nome'] / 'weights' / 'best.pt'))
    resultados_teste_por_experimento[config['nome']] = melhor_modelo.val(
        data=str(DATASET_YAML),
        split='test',
        imgsz=IMGSZ,
        plots=True,
        project=str(BASE_DIR / 'models'),
        name=f"val_{config['nome']}",
        exist_ok=True,
    )

## 5. Registro das métricas

Uma linha por experimento em `results/metrics.csv`; a tabela exibida traz todos os experimentos do projeto.

In [ ]:
import pandas as pd
from datetime import datetime

CSV_METRICAS = BASE_DIR / 'results' / 'metrics.csv'

def salvar_metricas(nome_experimento, metricas, caminho_csv):
    linha = {'experimento': nome_experimento, **metricas, 'data_hora': datetime.now().isoformat(timespec='seconds')}
    caminho_csv.parent.mkdir(parents=True, exist_ok=True)
    if caminho_csv.exists():
        tabela = pd.read_csv(caminho_csv)
        tabela = tabela[tabela['experimento'] != nome_experimento]
        tabela = pd.concat([tabela, pd.DataFrame([linha])], ignore_index=True)
    else:
        tabela = pd.DataFrame([linha])
    tabela.to_csv(caminho_csv, index=False)
    return tabela

for config in CONFIGURACOES:
    resultado = resultados_teste_por_experimento[config['nome']]
    metricas = {
        'mAP50': round(float(resultado.box.map50), 4),
        'mAP50_95': round(float(resultado.box.map), 4),
        'precision': round(float(resultado.box.mp), 4),
        'recall': round(float(resultado.box.mr), 4),
        'epochs': config['epochs'],
    }
    tabela = salvar_metricas(config['nome'], metricas, CSV_METRICAS)

tabela

## 6. Curvas comparadas: 20 vs 50 épocas

O gráfico sobrepõe o mAP@0,5 de validação por época dos dois runs. O que observar para o relatório:

- Se a curva de 50 épocas continua subindo e estabiliza, o orçamento era o gargalo (subtreino).
- Se ela atinge um pico e **cai** enquanto a perda de treino segue melhorando, é o início de overfitting - e o `best.pt` (selecionado pela validação) protege o resultado final.

O gráfico é salvo em `models/comparacao_epocas_map50.png` para uso no relatório.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

figura, eixo = plt.subplots(figsize=(10, 5))
for config in CONFIGURACOES:
    curva = pd.read_csv(BASE_DIR / 'models' / config['nome'] / 'results.csv')
    curva.columns = [c.strip() for c in curva.columns]
    eixo.plot(curva['epoch'], curva['metrics/mAP50(B)'], marker='o', markersize=3, label=f"{config['nome']} ({config['epochs']} épocas)")
eixo.set_xlabel('época')
eixo.set_ylabel('mAP@0.5 na validação')
eixo.legend()
eixo.grid(True, alpha=0.4)
figura.tight_layout()
figura.savefig(BASE_DIR / 'models' / 'comparacao_epocas_map50.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Ponto de operação para a anonimização

O mAP resume a qualidade média do detector, mas a implantação exige escolher **um limiar de confiança**. A validação da Ultralytics fornece as curvas precision×confiança e recall×confiança; a tabela abaixo mostra o compromisso em alguns limiares (a última linha é o ponto de **F1 máximo**).

Na anonimização, os dois erros têm custos assimétricos: rosto não detectado = dado pessoal exposto (grave); região borrada sem rosto = perda estética (leve). Por isso o ponto de operação adequado fica em confiança **baixa** (recall alto) - e não no F1 máximo, que trata os dois erros como equivalentes. Essa escolha justificada pertence à seção de avaliação ética do relatório.

In [ ]:
import numpy as np

NOME_FINAL = CONFIGURACOES[-1]['nome']
aval = resultados_teste_por_experimento[NOME_FINAL]
curvas = {nome: dados for nome, dados in zip(aval.curves, aval.curves_results)}
conf_x = np.asarray(curvas['Precision-Confidence(B)'][0])
precisao_y = np.asarray(curvas['Precision-Confidence(B)'][1])[0]
recall_y = np.asarray(curvas['Recall-Confidence(B)'][1])[0]
f1_y = np.asarray(curvas['F1-Confidence(B)'][1])[0]

linhas = []
for alvo in [0.05, 0.10, 0.25, 0.50]:
    i = int(np.argmin(np.abs(conf_x - alvo)))
    linhas.append({'conf': alvo, 'precision': round(float(precisao_y[i]), 3), 'recall': round(float(recall_y[i]), 3), 'F1': round(float(f1_y[i]), 3)})
i = int(np.argmax(f1_y))
linhas.append({'conf': round(float(conf_x[i]), 3), 'precision': round(float(precisao_y[i]), 3), 'recall': round(float(recall_y[i]), 3), 'F1': round(float(f1_y[i]), 3)})

pd.DataFrame(linhas)

## Conclusões dos experimentos

Pontos para fechar no relatório (e atualizar em `docs/experimentos.md`):

1. O exp4 confirmou a hipótese de subtreino do exp3? (compare com o exp2 no `metrics.csv`)
2. O exp5 mostrou ganho com mais épocas, saturação ou início de overfitting? Em que época a validação atingiu o melhor valor?
3. Qual limiar de confiança você recomendaria para a plataforma cívica, e por quê? (justifique pela assimetria de custos dos erros)
4. Qual é o modelo final recomendado do projeto, considerando recall como métrica crítica?